# Visualisation of the result 

In [ ]:
using DataFrames
import CSV
#Importation des données
file_path_df_final="pdb_information_details_final_1000.csv"
df=DataFrames.DataFrame(CSV.File(file_path_df_final,
comment="#", missingstring=["", "None"])) # Output DF with PDB CHAIN RESOLUTION SITE LIGAND
println(first(df,20))
println(size(df))

## Analyse global

In [ ]:
#Voir dispersion des dates de publication 

df.DATE_RELEASE = Date.(df.DATE_RELEASE, "yyyy-mm-dd")

histogram(year.(df.DATE_RELEASE), bins=10, color=:skyblue, edgecolor=:black, 
    title="Répartition des dates de publication", xlabel="Année de publication", ylabel="Nombre de structures")


In [ ]:
#Voir dispersion des résolutions
plot(df.RESOLUTION, 
    title="Répartition des resolutions", xlabel="Frequence", ylabel="Resolution")


In [ ]:
#Voir si lien entre resolutions et date de publication
scatter(df.DATE_RELEASE, df.RESOLUTION, color=:red, alpha=0.7, markerstrokewidth=0, 
    title="Résolution vs Date de publication", xlabel="Date de publication", ylabel="Résolution (Å)")
yflip!()  # En cristallographie, une valeur plus basse est meilleure

In [ ]:
#Voir la répartition entre les differentes méthodes
histogram(df.METHOD,
    title="Répartition des differentes méthodes", xlabel="Methodes", ylabel="Frequence")

In [ ]:
#Voir lien entre le méthode et la résolution 
# Filtrer les données pour exclure la méthode "NMR"
df_xray = filter(row -> row.METHOD != "NMR", df)

# Boxplot de la résolution en fonction de la méthode utilisée
@df df_xray boxplot(:METHOD, :RESOLUTION, color=:lightblue, outlier=:circle, 
    title="Comparaison des résolutions par méthode", xlabel="Méthode", ylabel="Résolution (Å)")
yflip!()  # Valeurs plus basses = meilleure résolution

In [ ]:
#Voir dispersion du nombes de domaine dans PFAM_NB
plot(df.PFAM_NB, 
    title="Répartition du nombbre de domaine PFAM_NB", xlabel="Frequence", ylabel="Resolution")

In [ ]:
#Voir dispersion du nombes de domaine dans CATH_NB
plot(df.CATH_NB, 
    title="Répartition du nombbre de domaine CATH_NB", xlabel="Frequence", ylabel="Resolution")

In [ ]:
#Voir dispersion du nombre de mutation
plot(df.MUTATION, 
    title="Répartition des mutations", xlabel="Frequence", ylabel="Resolution")

In [ ]:
#Voir dispersion du nombre de residues manquant
plot(df.MISSING_RESIDUES, 
    title="Répartition des missing residues", xlabel="Frequence", ylabel="Resolution")

## Analyse en fonction des formes APO et HOLO

In [ ]:
#Regroupe les pdb en fonction form apo et holo
df_grouped = combine(groupby(df, :LIGAND_PRESENT),
    nrow => :Count,
    :RESOLUTION => mean => :Mean_Resolution,
    :PFAM_NB => mean => :Mean_PFAM,
    :CATH_NB => mean => :Mean_CATH,
    :MUTATION => mean => :Mean_MUTATION,
    :MISSING_RESIDUES => mean => :Mean_MISSING_RESSIDUES
)
println(first(df_grouped,20))

In [ ]:
# Barplot pour comparer le nombre de structures
@df df_grouped bar(:LIGAND_PRESENT, :Count, legend=false, color=[:blue :red], 
    title="Comparaison du nombre de structures", xlabel="Présence de ligand", ylabel="Nombre de structures", xticks=([0,1], ["Apo", "Holo"]))


In [ ]:
# Boxplot de la résolution en fonction de la présence de ligand
@df df dropmissing boxplot(:LIGAND_PRESENT, :RESOLUTION, color=[:blue :red],
    title="Comparaison des résolutions", xlabel="Présence de ligand", ylabel="Résolution (Å)")
yflip!()  # La résolution plus basse est meilleure


In [ ]:
# Barplot pour comparer les annotations PFAM et CATH
@df df_grouped groupedbar(["Apo", "Holo"], [df_grouped.Mean_PFAM df_grouped.Mean_CATH], 
    bar_position=:dodge, color=[:blue :red], 
    title="Comparaison du nombre moyen de domaines", xlabel="Présence de ligand", ylabel="Nombre moyen de domaines", 
    label=["PFAM" "CATH"])

In [ ]:
# Barplot pour comparer les annotations PFAM et CATH
@df df_grouped groupedbar(["Apo", "Holo"], df_grouped.Mean_MUTATION, 
    bar_position=:dodge, color=[:blue :red], 
    title="Comparaison du nombre moyen de mutation ", xlabel="Présence de ligand", ylabel="Nombre moyen de domaines")

In [ ]:
# Barplot pour comparer le nombre de missing residues
@df df_grouped groupedbar(["Apo", "Holo"], df_grouped.Mean_MISSING_RESIDUES, 
    bar_position=:dodge, color=[:blue :red], 
    title="Comparaison du nombre moyen de resdidues manquant ", xlabel="Présence de ligand", ylabel="Nombre moyen de domaines")

## Analyse des clusters

In [ ]:
df_cluster_stats = DataFrame(
    Cutoff = ["1.5", "2.0", "4.0"],
    Mean_Clusters = [mean(df.Cluster_1.5), mean(df.Cluster_2.0), mean(df.Cluster_4.0)]
)
df_cluster_stats

In [ ]:
clusters = [:Cluster_1.5, :Cluster_2.0, :Cluster_4.0]

In [ ]:
# Barplot du nombre moyen de clusters par cutoff
@df df_cluster_stats bar(:Cutoff, :Mean_Clusters, legend=false, color=:blue,
    title="Nombre moyen de clusters par cutoff", xlabel="Cutoff", ylabel="Nombre moyen de clusters")


In [ ]:

function check_apo_holo_cluster(df_completed::DataFrame)
    check_cluster=DataFrame(UNIPROT=String[],HOLO_APO=Bool[],BEST_CUTOFF=Union{Int64,Missing}[])
    for row in eachrow(df_completed)
        uniprot=row.UNIPROT
        #println(uniprot)
        #Verifie qu'on a pas déja fait 
        is_present = uniprot in check_cluster.UNIPROT
        println(is_present)
        if !is_present
            df_uniprot = select(filter(row -> row.UNIPROT == uniprot, df_completed), ["PDB", "LIGANDS", "Cluster_1.5", "Cluster_2.0","Cluster_4.0"])# Pour ce centrer par uniprot et garder que information qui nous interesse 
            #println(df_uniprot)
            is_apo = any(ismissing, df_uniprot.LIGANDS)
            if is_apo 
                println(is_apo)
                # Filtrer les fichiers avec et sans ligands
                with_ligands = filter(row -> !ismissing(row.LIGANDS), df_uniprot)
                without_ligands = filter(row -> ismissing(row.LIGANDS), df_uniprot)
                result = 9

                # Comparer les clusters pour chaque cutoff
                for cutoff in [1.5, 2.0, 4.0]
                    # Récupérer les clusters pour chaque cutoff
                    cluster_col = Symbol("Cluster_$(cutoff)")
                    #println(cluster_col)
                    # Comparer les fichiers sans ligands et avec ligands
                    for file_with_ligands in eachrow(with_ligands)
                        for file_without_ligands in eachrow(without_ligands)
                            if file_with_ligands[cluster_col] != file_without_ligands[cluster_col]
                                # Si les fichiers sont dans des clusters différents, prendre la valeur la plus petite
                                if  cutoff<result
                                    result = cutoff
                                    println("oui")
                                end
                            end
                        end
                    end
                end
                push!(check_cluster,(uniprot,true,result))
            else 
                push!(check_cluster,(uniprot,false,missing))
            end 
        end
    end
    return check_cluster
end
check_cluster=check_apo_holo_cluster(df)
println(first(check_cluster,20))
most_frequent_value = mode(check_cluster.BEST_CUTOFF)
println("Le cutoff qui separe le mieux la forme apo et holo est : ", most_frequent_value)
frequency = count(x -> x == most_frequent_value, check_cluster.BEST_CUTOFF)
println("Elle apparaît ", frequency, " fois.")
println(size(check_cluster))
println("Elle separe correctement : ",(frequency/size(check_cluster)[1])*100)

In [ ]:
#Voir la repartition de Holo/APO
df_count = combine(groupby(df, :HOLO_APO), nrow => :Count)
@df df_count groupedbar( :Count, group=:HOLO_APO, bar_position=:dodge, 
                        xlabel="UniProt", ylabel="Nombre", title="Répartition HOLO/APO ", legend=:topleft)

In [ ]:
#Voir dispribution meilleur cutoff
@df df histogram(:BEST_CUTOFF, bins=5, xlabel="Best Cutoff", ylabel="Nombre", title="Distribution de Best Cutoff")